# DATMM consolidated research-graph notebook

One notebook: restore/import or acquire → cache/checkpoint → prioritize structural branches → enrich → reconstruct → validate/debug → export compact browser explorer.

Embedded cohort: **78 dbGaP datasets, 64 NCT identifiers, 88 associations** from the supplied CSV. No previous notebook, Python variables, or patch is required. Saved publication metadata is a source layer, not a reuse assertion.

Nine selectable columns: the eight native DATMM classes (Dataset, Collection, Agent, Concept, Contribution, Documentation, Grant, Repository) plus **Clinical study**, an application extension for registry records/references. Clinical study is not falsely asserted as a native DATMM class. Contribution is explicit; all Documentation genres and Concept schemes are retained.

Acquisition has no manual review gates; optional public publication is a separate explicit action. Source classifications and automated checks remain. A funding or publication association is not proof of causality/reuse. Source-local IDs and original triples are preserved.

The fixed defaultdict mutation uses non-mutating reads and snapshot iteration. A synthetic regression test is included. Structural Grant/Contribution/Agent/Repository branches have independent retrieval budgets so subjects/documents cannot starve them. Collection may remain zero; the browser can skip empty classes.

The browser has model-guided nine slots, presets, auto-updating facets with visible render counters/errors, strict adjacent-column edges, compact pan/zoom, click highlight, source links, resets, and optional NIH/NHLBI display aggregation. It works offline after export.

Documentation:
https://www.nlm.nih.gov/DATMM/datmm-vocab_03_model_overview.html
https://www.nlm.nih.gov/DATMM/datmm-vocab_04_vocab_specification.html

Generated code is syntax-checked and selected model helpers are regression-tested; no live acquisition or Drive writes are performed by the assistant.

**Release packaging:** Stored outputs have been cleared. The supplied browser snapshot is preserved separately at `docs/index.html`. Its original acquisition caches were not attached, so an offline rebuild from the embedded cohort alone will differ. See `RELEASE_GUIDE.md` and the final optional publication cell.


In [ ]:
# Pinned direct dependencies; not a reconstruction of the original runtime.
%pip -q install requests==2.32.5 pandas==2.2.3 plotly==5.24.1 ipython==8.37.0


In [ ]:
# @title
import json,re,time,hashlib,shutil,zipfile,base64,zlib,itertools
from pathlib import Path
from datetime import datetime,timezone,timedelta
from collections import defaultdict,Counter,deque
import xml.etree.ElementTree as ET
import requests,pandas as pd
from IPython.display import display
from plotly.offline import get_plotlyjs

MODE='cache_first'  # cache_first, offline, refresh_stale, full_refresh
try:
    import google.colab
    IN_COLAB=True
except ImportError:
    IN_COLAB=False
USE_DRIVE=IN_COLAB  # Set False to avoid Drive persistence in Colab.
DATASET_LIMIT=None  # None = all 78; e.g. 12 for a smaller run
FETCH_CATALOG=True
FETCH_STRUCTURAL=True
FETCH_DOCUMENTATION=True
FETCH_CONCEPTS=True
FETCH_TRIALS=True
FETCH_PUBMED=True
STALE_DAYS=30
BUDGETS={'rest':180,'seed_rdf':100,'structural':160,'documentation':60,'concept':20,'trial':70,'pubmed':20}
RDF_BATCH=6
RDF_LIMIT=500
STRUCTURAL_ROUNDS=5
CHECKPOINT_EVERY=30
WORK=Path('/content/datmm_next') if Path('/content').exists() else Path('datmm_next')
WORK.mkdir(parents=True,exist_ok=True)
ROOT=WORK/'consolidated';ROOT.mkdir(exist_ok=True)
OUT=ROOT/'output';OUT.mkdir(exist_ok=True)
CACHE=ROOT/'cache';CACHE.mkdir(exist_ok=True)
DRIVE_ROOT=None
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT=Path('/content/drive/MyDrive/DATMM_pilot')
    (DRIVE_ROOT/'checkpoints').mkdir(parents=True,exist_ok=True)
assert MODE in {'cache_first','offline','refresh_stale','full_refresh'}
CHECKS=[];HTTP=[];CALLS=Counter();TOTAL_CALLS=0

def check(name,ok=None,detail=''):
    state='SKIP' if ok is None else 'PASS' if ok else 'FAIL'
    CHECKS.append({'name':name,'status':state,'detail':str(detail)})
    print(f'[{state}] {name}: {detail}')
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def unpack(path,target):
    target=Path(target).resolve()
    with zipfile.ZipFile(path) as z:
        for info in z.infolist():
            p=(target/info.filename).resolve()
            if p!=target and target not in p.parents:raise ValueError('Unsafe ZIP path')
            if ((info.external_attr>>16)&0o170000)==0o120000:raise ValueError('ZIP symlink rejected')
        z.extractall(target)

# New runtime: restore newest compatible checkpoint, but do not overwrite an active consolidated cache.
if not (ROOT/'state.json').exists() and DRIVE_ROOT:
    backups=sorted((DRIVE_ROOT/'checkpoints').glob('datmm_checkpoint_*.zip'))
    if backups:
        local=WORK.parent/'consolidated_restore.zip';shutil.copy2(backups[-1],local);unpack(local,WORK)
        manifest=WORK/'checkpoint_manifest.json'
        if manifest.exists():
            for rel,expected in json.loads(manifest.read_text()).get('sha256',{}).items():
                p=(WORK/rel).resolve()
                if WORK.resolve() not in p.parents or not p.exists() or sha(p)!=expected:raise IOError('Restore checksum mismatch: '+rel)
        print('Restored:',backups[-1].name)
for p in (ROOT,OUT,CACHE):p.mkdir(exist_ok=True)

def checkpoint(label):
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    (ROOT/'state.json').write_text(json.dumps({'utc':stamp,'label':label,'new_source_calls':dict(CALLS)},indent=2))
    hashes={str(p.relative_to(WORK)):sha(p) for p in WORK.rglob('*') if p.is_file() and p.name!='checkpoint_manifest.json'}
    (WORK/'checkpoint_manifest.json').write_text(json.dumps({'utc':stamp,'label':label,'sha256':hashes},indent=2))
    archive=shutil.make_archive(str(WORK.parent/('datmm_checkpoint_'+stamp)),'zip',root_dir=WORK)
    if DRIVE_ROOT:
        dest=DRIVE_ROOT/'checkpoints'/Path(archive).name;temp=dest.with_suffix('.partial');shutil.copy2(archive,temp)
        if sha(temp)!=sha(archive):raise IOError('Drive checkpoint mismatch')
        temp.replace(dest)
    print('Checkpoint:',Path(archive).name);return archive

_original=getattr(requests.sessions.Session,'_datmm_original_request',requests.sessions.Session.request)
session=requests.Session();session.headers['User-Agent']='DATMM-Consolidated-Colab/1.0'

def fresh(entry):return datetime.now(timezone.utc)-datetime.fromisoformat(entry['utc'])<=timedelta(days=STALE_DAYS)
def cached_call(source,label,method,url,validate,**kwargs):
    global TOTAL_CALLS
    key=hashlib.sha256(json.dumps([method,url,kwargs],sort_keys=True).encode()).hexdigest()
    folder=CACHE/source;folder.mkdir(exist_ok=True);cp=folder/(key+'.json')
    old=json.loads(cp.read_text()) if cp.exists() else None
    if old and MODE!='full_refresh' and (MODE!='refresh_stale' or fresh(old)):return old
    if MODE=='offline' or CALLS[source]>=BUDGETS[source]:
        if old:return old
        raise RuntimeError('Cache miss / offline / budget: '+source+' '+label)
    CALLS[source]+=1;TOTAL_CALLS+=1;time.sleep(.4)
    item={'source':source,'label':label,'utc':datetime.now(timezone.utc).isoformat(),'method':method,'url':url,'request':kwargs}
    try:
        response=_original(session,method,url,timeout=(10,45),**kwargs)
        item.update(status=response.status_code,text=response.text);response.raise_for_status()
        try:item['json']=response.json()
        except ValueError:item['json']=None
        if not validate(item):raise ValueError('Invalid response shape: '+label)
        if old:
            history=folder/'history';history.mkdir(exist_ok=True)
            shutil.copy2(cp,history/(key+'_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')+'.json'))
        temp=cp.with_suffix('.partial');temp.write_text(json.dumps(item,ensure_ascii=False));temp.replace(cp)
        HTTP.append({k:v for k,v in item.items() if k not in ('text','json')});return item
    except (requests.RequestException,ValueError) as exc:
        item['error']=str(exc);(folder/(key+'_failure.json')).write_text(json.dumps(item,ensure_ascii=False))
        HTTP.append({k:v for k,v in item.items() if k not in ('text','json')})
        if old:
            check('Cached response fallback',None,label+' refresh failed; old timestamp retained');return old
        raise

In [ ]:
# @title
COHORT=json.loads(zlib.decompress(base64.b64decode('...')).decode())
if DATASET_LIMIT is not None:COHORT=COHORT[:DATASET_LIMIT]
NCTS=sorted({n for s in COHORT for n in s['ncts']})
PAIRS=[{'accession':s['accession'],'nct':n} for s in COHORT for n in s['ncts']]
print('Active cohort:',len(COHORT),'datasets;',len(NCTS),'clinical studies;',len(PAIRS),'associations')
(OUT/'cohort.json').write_text(json.dumps(COHORT,ensure_ascii=False))
pd.DataFrame(PAIRS).to_csv(OUT/'seed_associations.csv',index=False)

## A. Load evidence or acquire missing catalog records
Existing full-cohort/recovery triples are imported. Missing work is retrieved when enabled. Reads never mutate the index dictionary. Exact-version accession checks are required for catalog mappings; source seed records remain visible even if catalog mapping fails.

Complete refreshed subject responses replace that subject's old triples in the active index; historical checkpoints/cache copies remain. Incomplete responses are marked and combined conservatively, not declared complete.

In [ ]:
# @title
RDF_TYPE='http://www.w3.org/1999/02/22-rdf-syntax-ns#type'
TYPE_URIS={
 'Dataset':{'http://id.nlm.nih.gov/datmm/Dataset','https://id.nlm.nih.gov/datmm/Dataset'},
 'Collection':{'http://purl.org/dc/dcmitype/Collection'},'Agent':{'http://xmlns.com/foaf/0.1/Agent'},
 'Concept':{'http://www.w3.org/2004/02/skos/core#Concept'},'Contribution':{'http://id.loc.gov/ontologies/bibframe/Contribution'},
 'Documentation':{'http://id.nlm.nih.gov/datmm/Documentation','https://id.nlm.nih.gov/datmm/Documentation'},
 'Grant':{'http://schema.org/Grant','https://schema.org/Grant'},'Repository':{'http://id.nlm.nih.gov/datmm/Repository','https://id.nlm.nih.gov/datmm/Repository'}}
CLASS_NAMES=list(TYPE_URIS)+['Clinical study']
raw_options=[OUT/'raw_triples.json',WORK/'full_cohort'/'output'/'raw_triples.json',WORK/'unified_explorer'/'raw_triples.json',WORK/'state'/'raw_triples.json']
raw_file=next((p for p in raw_options if p.exists()),None)
RAW=json.loads(raw_file.read_text()) if raw_file else []
INDEX={}
for t in RAW:
    if all(k in t for k in ('s','p','o')):INDEX.setdefault(t['s'],[]).append(t)
print('Imported RDF rows:',len(RAW),'subjects:',len(INDEX))

def local(v):return str(v).rsplit('/',1)[-1].rsplit('#',1)[-1]
def values(uri,prop,index=None):
    idx=INDEX if index is None else index
    return [t['o'] for t in idx.get(uri,[]) if local(t['p'])==prop]
def recognized(uri,index=None):
    idx=INDEX if index is None else index
    types={t['o'] for t in idx.get(uri,[]) if t['p']==RDF_TYPE}
    return {k for k,uris in TYPE_URIS.items() if types&uris}
def label(uri,kind='',index=None):
    idx=INDEX if index is None else index
    if kind=='Dataset':
        ids=[x for x in values(uri,'identifier',idx) if re.fullmatch(r'phs\d+(?:\.v\d+\.p\d+)?',x)]
        if ids:return ids[0]
    for name in ('prefLabel','title','name','label','identifier'):
        vals=values(uri,name,idx)
        if vals:return vals[0]
    return local(uri)

# Regression test: labeling an absent role must not add dictionary entries.
fixture={'urn:test:contribution':[{'s':'urn:test:contribution','p':'http://id.loc.gov/ontologies/bibframe/role','o':'urn:missing:role','o_type':'uri'}]}
before=len(fixture)
for uri,ts in list(fixture.items()):
    for role in values(uri,'role',fixture):label(role,'Role',fixture)
assert len(fixture)==before
check('Non-mutating role lookup regression',True,'Absent role lookup did not change dictionary size')

ENTITY=CACHE/'entities';ENTITY.mkdir(exist_ok=True)
CAPPED=set();FAILED=set();RESOLVED={};SEED_STATUS=[]
ENDPOINT='https://xvonuqz0ik.execute-api.us-east-1.amazonaws.com/prod/api/sparql/'
def safeiri(uri):
    if not uri.startswith(('http://','https://')) or any(c.isspace() for c in uri) or any(c in uri for c in '<>"{}|^`'):raise ValueError('Unsafe observed URI')
    return '<'+uri+'>'
def valid_rdf(item):
    p=item.get('json');return isinstance(p,dict) and isinstance(p.get('head',{}).get('vars'),list) and isinstance(p.get('results',{}).get('bindings'),list)
def save_raw():
    rows=[t for ts in INDEX.values() for t in ts];temp=OUT/'raw_triples.partial';temp.write_text(json.dumps(rows,ensure_ascii=False));temp.replace(OUT/'raw_triples.json')

def apply_entity(uri,rows,complete):
    if complete:INDEX[uri]=rows
    else:
        CAPPED.add(uri);existing=INDEX.setdefault(uri,[])
        keys={json.dumps(t,sort_keys=True) for t in existing}
        for t in rows:
            if json.dumps(t,sort_keys=True) not in keys:existing.append(t)

def rdf(uris,source,label_text):
    missing=[]
    for uri in uris:
        cp=ENTITY/(hashlib.sha256(uri.encode()).hexdigest()+'.json')
        prior=json.loads(cp.read_text()) if cp.exists() else None
        if prior and prior.get('complete') and MODE!='full_refresh' and (MODE!='refresh_stale' or fresh(prior)):
            apply_entity(uri,prior['triples'],True)
        else:missing.append(uri)
    if not missing:return
    q=f"SELECT ?s ?p ?o WHERE {{ VALUES ?s {{ {' '.join(safeiri(u) for u in missing)} }} ?s ?p ?o }} LIMIT {RDF_LIMIT}"
    result=cached_call(source,label_text,'POST',ENDPOINT,valid_rdf,json={'query_type':'raw_sparql','raw_sparql':q})
    bindings=result['json']['results']['bindings'];complete=len(bindings)<RDF_LIMIT
    if not complete and len(missing)>1:
        middle=len(missing)//2;rdf(missing[:middle],source,label_text+'-a');rdf(missing[middle:],source,label_text+'-b');return
    grouped=defaultdict(list)
    for b in bindings:
        if not all(k in b for k in ('s','p','o')) or b['s'].get('type')!='uri':continue
        obj=b['o']['value']
        if b['o'].get('type')=='bnode':obj=label_text+'::'+obj
        grouped[b['s']['value']].append({'s':b['s']['value'],'p':b['p']['value'],'o':obj,'o_type':b['o'].get('type'),
            'datatype':b['o'].get('datatype'),'language':b['o'].get('xml:lang'),'query':q,'retrieved_utc':result['utc']})
    for uri in missing:
        entry={'uri':uri,'utc':result['utc'],'complete':complete,'triples':grouped[uri]}
        (ENTITY/(hashlib.sha256(uri.encode()).hexdigest()+'.json')).write_text(json.dumps(entry,ensure_ascii=False))
        apply_entity(uri,grouped[uri],complete)
    if TOTAL_CALLS and TOTAL_CALLS%CHECKPOINT_EVERY==0:save_raw();checkpoint('incremental-rdf')

KNOWN={'phs004742':'http://id.nlm.nih.gov/datmm/dataset/0003237541','phs003470':'http://id.nlm.nih.gov/datmm/dataset/0002942769',
       'phs002377':'http://id.nlm.nih.gov/datmm/dataset/0002943560','phs003291':'http://id.nlm.nih.gov/datmm/dataset/0002944665',
       'phs002486':'http://id.nlm.nih.gov/datmm/dataset/0002942685','phs004618':'http://id.nlm.nih.gov/datmm/dataset/0003016429'}
mapping_path=ROOT/'catalog_mapping.json'
mapping=json.loads(mapping_path.read_text()) if mapping_path.exists() else {}
for seed in COHORT:
    acc=seed['accession'];candidates=[u for u in INDEX if 'Dataset' in recognized(u) and acc in values(u,'identifier')]
    if len(candidates)==1 and MODE in {'cache_first','offline'}:
        RESOLVED[acc]=candidates[0];state='exact_version_cached'
    elif FETCH_CATALOG:
        try:
            if not candidates:
                candidates=[mapping[acc]] if acc in mapping else [KNOWN[seed['base']]] if seed['base'] in KNOWN else []
            if not candidates:
                for search in (seed['base'],seed['title'][:190]):
                    item=cached_call('rest','resolve-'+seed['base'],'GET','https://datasetcatalog.nlm.nih.gov/public/api/v1',
                        lambda e:isinstance(e.get('json'),dict) and isinstance(e['json'].get('results'),list),params={'q':search,'size':50,'page':1})
                    for hit in item['json']['results']:
                        vals=[hit.get('source',{}).get('homepage',''),hit.get('source',{}).get('identifier','')]
                        if hit.get('id') and any(re.search(r'(?<![A-Za-z0-9])'+re.escape(seed['base'])+r'(?!\d)',str(v),re.I) for v in vals):candidates.append(hit['id'])
                    if candidates:break
            matches=[]
            for uri in list(dict.fromkeys(candidates))[:3]:
                rdf([uri],'seed_rdf','seed-'+seed['base'])
                if 'Dataset' in recognized(uri) and acc in values(uri,'identifier'):matches.append(uri)
            if len(set(matches))==1:RESOLVED[acc]=matches[0];mapping[acc]=matches[0];state='exact_version_retrieved'
            else:state='unresolved_or_ambiguous_version'
        except (RuntimeError,requests.RequestException,ValueError) as exc:state='retrieval_failed: '+str(exc)
    else:state='acquisition_disabled'
    SEED_STATUS.append({'accession':acc,'uri':RESOLVED.get(acc),'status':state})
    mapping_path.write_text(json.dumps(mapping,indent=2))
pd.DataFrame(SEED_STATUS).to_csv(OUT/'dataset_mapping_coverage.csv',index=False)
check('Exact-version catalog mappings',len(RESOLVED)==len(COHORT),f'{len(RESOLVED)}/{len(COHORT)}; unresolved seed entities retained as source-layer datasets')

## B. Structural recovery gets its own budget
Grant, Contribution, Agent, Repository and collection/member branches are completed before additional subjects or documentation. Each pass uses only observed target URIs. The original large concept/documentation branch cannot consume the structural request budget. Zero Collection population is not fixed by inventing a class instance.

In [ ]:
# @title
STRUCTURAL={
 'http://schema.org/funding':0,'https://schema.org/funding':0,
 'http://id.loc.gov/ontologies/bibframe/contribution':1,
 'http://schema.org/funder':2,'https://schema.org/funder':2,
 'http://id.loc.gov/ontologies/bibframe/agent':3,
 'http://purl.org/dc/terms/isPartOf':4,'http://purl.org/dc/terms/hasPart':4,
 'http://purl.org/dc/terms/isVersionOf':5,'http://purl.org/dc/terms/relation':5}
DOC_PREDICATES={'http://purl.org/dc/terms/isReferencedBy','http://purl.org/dc/terms/references'}
SUBJECT_PREDICATES={'http://purl.org/dc/terms/subject'}

def missing_targets(predicates,seen):
    priorities={}
    for ts in list(INDEX.values()):
        for t in ts:
            if t['p'] in predicates and t.get('o_type')=='uri' and t['o'] not in seen and not recognized(t['o']):
                priority=predicates[t['p']] if isinstance(predicates,dict) else 0
                priorities[t['o']]=min(priorities.get(t['o'],99),priority)
    return sorted(priorities,key=lambda u:(priorities[u],u))

def expand(source,predicates,enabled,rounds):
    seen=set()
    if not enabled:return
    for round_id in range(rounds):
        targets=missing_targets(predicates,seen)
        if not targets:break
        print(source,'round',round_id+1,'untyped targets:',len(targets))
        for i in range(0,len(targets),RDF_BATCH):
            batch=targets[i:i+RDF_BATCH];seen.update(batch)
            try:rdf(batch,source,source+'-'+str(round_id)+'-'+str(i))
            except (RuntimeError,requests.RequestException,ValueError):FAILED.update(batch)
            if MODE!='offline' and CALLS[source]>=BUDGETS[source]:break
        if MODE!='offline' and CALLS[source]>=BUDGETS[source]:break

expand('structural',STRUCTURAL,FETCH_STRUCTURAL,STRUCTURAL_ROUNDS)
expand('documentation',DOC_PREDICATES,FETCH_DOCUMENTATION,1)
expand('concept',SUBJECT_PREDICATES,FETCH_CONCEPTS,1)
save_raw()
NATIVE_CLASS_COUNTS={kind:sum(kind in recognized(u) for u in INDEX) for kind in TYPE_URIS}
display(pd.DataFrame([{'class':k,'native_typed_subjects':v} for k,v in NATIVE_CLASS_COUNTS.items()]))
checkpoint('typed-branch-acquisition')

## C. Cached registry and PubMed cross-checks
Trial records and PubMed XML are fetched only as configured. Prior full-cohort/vNext record caches are imported by identifier when available. BACKGROUND/RESULT/DERIVED are preserved. Abstract mentions and structured PubMed accessions remain separate evidence. No publication acknowledgment becomes dataset funding.

In [ ]:
# @title
TRIALDIR=CACHE/'trial_records';TRIALDIR.mkdir(exist_ok=True)
PUBDIR=CACHE/'pubmed_records';PUBDIR.mkdir(exist_ok=True)
TRIALS={};PUBS={}
for nct in NCTS:
    cp=TRIALDIR/(nct+'.json');prior=json.loads(cp.read_text()) if cp.exists() else None
    if prior is None:
        olds=[WORK/'full_cohort'/'cache'/'trial_records'/(nct+'.json'),WORK/'cache'/'trials'/(nct+'.json')]
        for old in olds:
            if old.exists():
                data=json.loads(old.read_text());stamp=data.get('utc') or data.get('retrieved_utc')
                if stamp and data.get('record'):prior={'utc':stamp,'record':data['record']};cp.write_text(json.dumps(prior,ensure_ascii=False));break
    need=prior is None or MODE=='full_refresh' or (MODE=='refresh_stale' and not fresh(prior))
    if need and FETCH_TRIALS:
        try:
            item=cached_call('trial',nct,'GET','https://clinicaltrials.gov/api/v2/studies/'+nct,
                lambda e,n=nct:isinstance(e.get('json'),dict) and e['json'].get('protocolSection',{}).get('identificationModule',{}).get('nctId')==n)
            prior={'utc':item['utc'],'record':item['json']};cp.write_text(json.dumps(prior,ensure_ascii=False))
        except (RuntimeError,requests.RequestException,ValueError):pass
    if prior:TRIALS[nct]=prior
    if TOTAL_CALLS and TOTAL_CALLS%CHECKPOINT_EVERY==0:checkpoint('incremental-trials')

# Native/saved/registry PMIDs define a bounded literature universe; no free-text search expansion.
PMIDS={p['pmid'] for s in COHORT for p in s['publications'] if re.fullmatch(r'\d+',p['pmid'])}
for entry in TRIALS.values():
    for ref in entry['record'].get('protocolSection',{}).get('referencesModule',{}).get('references',[]):
        p=str(ref.get('pmid',''))
        if re.fullmatch(r'\d+',p):PMIDS.add(p)
for ts in INDEX.values():
    for t in ts:
        m=re.search(r'https?://(?:www\.)?pubmed\.ncbi\.nlm\.nih\.gov/(\d+)',t['o'])
        if m:PMIDS.add(m.group(1))
missing=[]
for p in sorted(PMIDS):
    cp=PUBDIR/(p+'.json');prior=json.loads(cp.read_text()) if cp.exists() else None
    if prior is None:
        for old in [WORK/'full_cohort'/'cache'/'pubmed_records'/(p+'.json'),WORK/'cache'/'pubmed'/(p+'.json')]:
            if old.exists():
                item=json.loads(old.read_text());record=item.get('record',item);stamp=item.get('utc') or record.get('retrieved_utc')
                if stamp:prior={'utc':stamp,'record':record};cp.write_text(json.dumps(prior,ensure_ascii=False));break
    if prior and MODE!='full_refresh' and (MODE!='refresh_stale' or fresh(prior)):PUBS[p]=prior
    else:missing.append(p)
def text(n):return ''.join(n.itertext()).strip() if n is not None else ''
def valid_xml(e):
    try:return ET.fromstring(e['text']).tag=='PubmedArticleSet'
    except ET.ParseError:return False
if FETCH_PUBMED:
    for i in range(0,len(missing),50):
        batch=missing[i:i+50]
        try:
            item=cached_call('pubmed','batch-'+str(i),'GET','https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi',valid_xml,
                params={'db':'pubmed','id':','.join(batch),'retmode':'xml','tool':'DATMMconsolidated'})
            root=ET.fromstring(item['text'])
            for article in root.findall('./PubmedArticle'):
                c=article.find('./MedlineCitation');p=text(c.find('./PMID'))
                if p not in batch:continue
                record={'pmid':p,'title':text(c.find('./Article/ArticleTitle')),
                        'abstract':'\n'.join(text(a) for a in c.findall('./Article/Abstract/AbstractText')),
                        'databanks':[],'grants':[],'retrieved_utc':item['utc']}
                for bank in c.findall('./Article/DataBankList/DataBank'):
                    for a in bank.findall('./AccessionNumberList/AccessionNumber'):record['databanks'].append({'database':text(bank.find('./DataBankName')),'accession':text(a)})
                for g in c.findall('./Article/GrantList/Grant'):record['grants'].append({'grant_id':text(g.find('./GrantID')),'agency':text(g.find('./Agency'))})
                prior={'utc':item['utc'],'record':record};(PUBDIR/(p+'.json')).write_text(json.dumps(prior,ensure_ascii=False));PUBS[p]=prior
        except (RuntimeError,requests.RequestException,ValueError,ET.ParseError):pass
        if CALLS['pubmed']>=BUDGETS['pubmed'] and MODE!='offline':break
check('Clinical-study coverage',len(TRIALS)==len(NCTS),f'{len(TRIALS)}/{len(NCTS)}')
check('PubMed coverage',len(PUBS)==len(PMIDS),f'{len(PUBS)}/{len(PMIDS)}')
pd.DataFrame([{'nct':n,'available':n in TRIALS} for n in NCTS]).to_csv(OUT/'trial_coverage.csv',index=False)
pd.DataFrame([{'pmid':p,'available':p in PUBS} for p in sorted(PMIDS)]).to_csv(OUT/'publication_coverage.csv',index=False)
checkpoint('external-source-enrichment')

## D. Model-native graph plus source extensions
All reads use INDEX.get; iteration is frozen with list(INDEX.items()). Contribution nodes remain distinct. Agent identity is a source URI, never a personal-name merge. Exact versioned accession, explicit MeSH identifier, and explicit PubMed homepage normalize identifiers while retaining aliases/triples.

Clinical study is a separate application class. Unresolved dbGaP records remain source-layer Dataset entities, visibly classified as not RDF-validated. Documentation includes all genres. Source cohort associations, native RDF, registry references and publication text are separately switchable.

In [ ]:
# @title
N={};E=[];ALIASES={};EDGE_SEEN=set()

def node(key,kind,lab,basis,alias=None,metadata=None,subtype=''):
    N.setdefault(key,{'id':key,'kind':kind,'label':lab,'classification_basis':basis,'aliases':[],'metadata':[],'subtype':subtype,'datasets':[]})
    if alias and alias not in N[key]['aliases']:N[key]['aliases'].append(alias)
    if metadata:
        if isinstance(metadata,list):N[key]['metadata'].extend(metadata)
        else:N[key]['source_metadata']=metadata
    if basis=='observed native RDF type':N[key]['classification_basis']=basis
    return key

def edge(s,o,relation,layer,support,predicate=None,seed_provenance=None):
    key=json.dumps([s,o,relation,layer,support],sort_keys=True)
    if key in EDGE_SEEN:return
    EDGE_SEEN.add(key);E.append({'s':s,'o':o,'relation':relation,'layer':layer,'predicate':predicate,'support':support,'source_dataset_provenance':seed_provenance or []})

for uri,ts in list(INDEX.items()):
    kinds=recognized(uri)
    if len(kinds)!=1:continue
    kind=next(iter(kinds));key=uri;subtype='';lab=label(uri,kind)
    if kind=='Dataset':
        ids=[v for v in values(uri,'identifier') if re.fullmatch(r'phs\d+\.v\d+\.p\d+',v)]
        if len(set(ids))==1:key='dbgap:'+ids[0];lab=ids[0]
    elif kind=='Documentation':
        pids={m.group(1) for t in ts for m in [re.search(r'https?://(?:www\.)?pubmed\.ncbi\.nlm\.nih\.gov/(\d+)',t['o'])] if m}
        if len(pids)==1:key='pmid:'+next(iter(pids));subtype='Publication'
        else:subtype='; '.join(values(uri,'type')) or 'Unspecified documentation'
    elif kind=='Concept':
        meshids={m.group(1) for v in [uri]+[t['o'] for t in ts] for m in [re.search(r'https?://id\.nlm\.nih\.gov/mesh/([DCQ]\d+)(?![A-Za-z0-9])',v)] if m}
        if len(meshids)==1:key='mesh:'+next(iter(meshids));subtype='MeSH'
        else:subtype='; '.join(values(uri,'inScheme')) or 'Local/unspecified concept scheme'
    elif kind=='Contribution':
        roles=[label(r,'Role') for r in values(uri,'role')];names=[label(a,'Agent') for a in values(uri,'agent')]
        subtype='; '.join(roles) or 'Role unspecified';lab=subtype+(' — '+names[0] if names else ' — '+local(uri))
    node(key,kind,lab,'observed native RDF type',uri,ts,subtype);ALIASES[uri]=key

for uri,ts in list(INDEX.items()):
    for t in ts:
        if t.get('o_type')=='uri' and t['s'] in ALIASES and t['o'] in ALIASES and t['p']!=RDF_TYPE:
            edge(ALIASES[t['s']],ALIASES[t['o']],local(t['p']),'native_datmm',[t],t['p'])

SEED_IDS=[]
for seed in COHORT:
    d=node('dbgap:'+seed['accession'],'Dataset',seed['accession'],'uploaded dbGaP seed; catalog identity not asserted',metadata=seed)
    SEED_IDS.append(d)
    for nct in seed['ncts']:
        n=node('nct:'+nct,'Clinical study',nct,'external registry/identifier evidence',subtype='Clinical-study reference')
        edge(d,n,'association in saved dbGaP extraction','uploaded_dataset_trial',[{'accession':seed['accession'],'nct':nct,'source':seed['source']}],seed_provenance=[d])
    for p in seed['publications']:
        if not re.fullmatch(r'\d+',p['pmid']):continue
        k=node('pmid:'+p['pmid'],'Documentation',p['title'] or 'PMID '+p['pmid'],'external publication evidence',subtype='Publication')
        edge(d,k,'publication listed by dbGaP','uploaded_dataset_publication',[p],seed_provenance=[d])
        for nct in set(re.findall(r'\bNCT\d{8}\b',p['notes'] or '',re.I)):
            nct=nct.upper()
            if nct in seed['ncts']:edge(k,'nct:'+nct,'NCT in saved publication notes','uploaded_publication_notes',[p],seed_provenance=[d])
    uri=RESOLVED.get(seed['accession'])
    if uri:
        for t in INDEX.get(uri,[]):
            if t.get('o_type') in ('literal','typed-literal'):
                for nct in set(re.findall(r'\bNCT\d{8}\b',t['o'],re.I)):
                    nct=nct.upper()
                    if nct in NCTS:edge(d,'nct:'+nct,'NCT mention in '+local(t['p']),'catalog_text_mention',[t],seed_provenance=[d])

for nct,entry in TRIALS.items():
    record=entry['record'];protocol=record.get('protocolSection',{});ident=protocol.get('identificationModule',{})
    k=node('nct:'+nct,'Clinical study',nct,'external registry/identifier evidence')
    N[k].update(label=nct+': '+ident.get('briefTitle',''),source_metadata=record,subtype=protocol.get('designModule',{}).get('studyType','Registered study'))
    ds=['dbgap:'+s['accession'] for s in COHORT if nct in s['ncts']]
    for ref in protocol.get('referencesModule',{}).get('references',[]):
        p=str(ref.get('pmid',''))
        if not re.fullmatch(r'\d+',p):continue
        pub=node('pmid:'+p,'Documentation',ref.get('citation') or 'PMID '+p,'external publication evidence',subtype='Publication')
        typ=ref.get('type','UNSPECIFIED');rel={'BACKGROUND':'background reference','RESULT':'registry-classified results reference','DERIVED':'derived publication association'}.get(typ,'registry reference')
        edge(pub,k,rel,'registry_reference',[{'reference':ref,'registry_type':typ,'utc':entry['utc']}],seed_provenance=ds)
    for term in record.get('derivedSection',{}).get('conditionBrowseModule',{}).get('meshes',[]):
        if re.fullmatch(r'[DCQ]\d+',term.get('id','')):
            concept=node('mesh:'+term['id'],'Concept',term.get('term') or term['id'],'external controlled subject evidence',subtype='MeSH')
            edge(concept,k,'registry condition MeSH','registry_mesh',[term],seed_provenance=ds)
for p,entry in PUBS.items():
    record=entry['record'];k=node('pmid:'+p,'Documentation',record.get('title') or 'PMID '+p,'external publication evidence',subtype='Publication')
    N[k].update(label=record.get('title') or N[k]['label'],pubmed_metadata=record)
    for nct in set(re.findall(r'\bNCT\d{8}\b',record.get('abstract',''),re.I)):
        nct=nct.upper()
        if nct in NCTS:edge(k,'nct:'+nct,'NCT in PubMed abstract','pubmed_abstract_mention',[record])
    for bank in record.get('databanks',[]):
        nct=bank['accession'].upper()
        if nct in NCTS:edge(k,'nct:'+nct,'structured PubMed NCT identifier','pubmed_structured_identifier',[bank])

# Merge repeated logical edge endpoints while preserving every evidence record.
logical={}
for e in E:
    k=(e['s'],e['o'],e['relation'],e['layer'])
    if k not in logical:logical[k]=e
    else:
        logical[k]['support'].extend(e['support']);logical[k]['source_dataset_provenance']=sorted(set(logical[k]['source_dataset_provenance'])|set(e['source_dataset_provenance']))
E=list(logical.values())
# Bounded dataset association for facets. Do not bleed through repository/agent/subject hubs to unrelated datasets.
adj=defaultdict(set)
for e in E:adj[e['s']].add(e['o']);adj[e['o']].add(e['s'])
all_datasets={u for u,n in N.items() if n['kind']=='Dataset'}
lineage=defaultdict(set);leaf={'Agent','Concept','Repository','Clinical study'}
for start in SEED_IDS:
    queue=deque([(start,0)]);seen={start}
    while queue:
        u,depth=queue.popleft();lineage[u].add(start)
        if depth>=4 or (u!=start and N[u]['kind'] in leaf):continue
        for nxt in adj.get(u,set()):
            if nxt in seen or (nxt in all_datasets and nxt!=start):continue
            seen.add(nxt);queue.append((nxt,depth+1))
for e in E:
    e['datasets']=sorted(lineage[e['s']]|lineage[e['o']]|set(e['source_dataset_provenance']))
for u,n in N.items():n['datasets']=sorted({u} if u in SEED_IDS else lineage[u])
GROUPS={}
for u,n in N.items():
    if n['kind']!='Agent':continue
    norm=re.sub(r'[^a-z0-9 ]',' ',n['label'].lower());norm=re.sub(r'\s+',' ',norm).strip()
    if norm.startswith('national institutes of health'):g,lab='display-agent:NIH','National Institutes of Health'
    elif norm.startswith('national heart') and 'lung' in norm and 'blood institute' in norm:g,lab='display-agent:NHLBI','National Heart, Lung, and Blood Institute'
    else:continue
    GROUPS[u]={'group':g,'label':lab,'source_uri':u,'rule':'organizational display aggregation; not verified identity equivalence'}
ARCH=[('Agent','Contribution'),('Contribution','Dataset'),('Contribution','Collection'),('Contribution','Grant'),
      ('Dataset','Grant'),('Dataset','Documentation'),('Dataset','Concept'),('Dataset','Repository'),('Dataset','Collection'),
      ('Collection','Repository'),('Collection','Documentation'),('Collection','Concept'),('Repository','Concept'),
      ('Dataset','Clinical study'),('Documentation','Clinical study'),('Concept','Clinical study')]
GRAPH={'nodes':list(N.values()),'edges':E,'classes':CLASS_NAMES,'architecture':ARCH,'organization_display_groups':GROUPS,
       'datasets':SEED_IDS,'created_utc':datetime.now(timezone.utc).isoformat()}
assert all(e['s'] in N and e['o'] in N for e in E)
assert all(n['kind'] in CLASS_NAMES for n in N.values())
check('Graph endpoint integrity',True,f'{len(N)} nodes; {len(E)} logical evidence edges')
check('Nine-class vocabulary integrity',True,'8 native classes + Clinical study application extension')
(OUT/'graph.json').write_text(json.dumps(GRAPH,ensure_ascii=False))
coverage=pd.DataFrame([{'class':c,'native_normalized_instances':sum(n['kind']==c and n['classification_basis']=='observed native RDF type' for n in N.values()),
                       'source_extension_instances':sum(n['kind']==c and n['classification_basis']!='observed native RDF type' for n in N.values())} for c in CLASS_NAMES])
display(coverage);coverage.to_csv(OUT/'class_coverage.csv',index=False)
availability=pd.DataFrame([{'source_class':N[e['s']]['kind'],'target_class':N[e['o']]['kind'],'relation':e['relation'],'layer':e['layer']} for e in E])
if not availability.empty:availability=availability.groupby(['source_class','target_class','relation','layer']).size().rename('edges').reset_index()
availability.to_csv(OUT/'edge_availability.csv',index=False)

## E. Shared projections and technical checks
Shared projections are overlap views, not native Dataset relations. Source edge IDs point back to the underlying evidence. Projection pairs are capped for portability; full graph and coverage remain exported. Dataset A/B are display roles of the same canonical entities.

Checks include dictionary-lookup regression, dangling endpoints, source coverage, class population, missing structural targets, source agreement and zero browser-source requests by design. Collection zero is an availability finding, not a conformance failure.

In [ ]:
# @title
for i,e in enumerate(E):e['edge_id']='edge-'+str(i)
HUBS=defaultdict(lambda:defaultdict(set))
seed_set=set(SEED_IDS)
for e in E:
    if e['layer'] not in {'native_datmm','uploaded_dataset_publication'}:continue
    for d,h in ((e['s'],e['o']),(e['o'],e['s'])):
        if d in seed_set and N[h]['kind'] in {'Concept','Documentation','Grant'}:HUBS[h][d].add(e['edge_id'])
# Actual model paths connect datasets to agents; projection is explicitly derived.
contrib_agents=defaultdict(list)
for e in E:
    if e['layer']=='native_datmm' and N[e['s']]['kind']=='Contribution' and N[e['o']]['kind']=='Agent':contrib_agents[e['s']].append(e)
for e in E:
    if e['layer']!='native_datmm':continue
    if e['s'] in seed_set and N[e['o']]['kind']=='Contribution':
        for a in contrib_agents[e['o']]:HUBS[a['o']][e['s']].update([e['edge_id'],a['edge_id']])
    if N[e['s']]['kind']=='Grant' and N[e['o']]['kind']=='Contribution':
        for d,ids in HUBS.get(e['s'],{}).items():
            for a in contrib_agents[e['o']]:HUBS[a['o']][d].update(ids|{e['edge_id'],a['edge_id']})
PROJECTIONS={};PAIR_CAP=1200
for kind in ('Documentation','Concept','Grant','Agent'):
    nn={};ee=[];count=0
    for h,ds in HUBS.items():
        if N[h]['kind']!=kind or len(ds)<2:continue
        nn[h]=N[h]
        for left,right in itertools.combinations(sorted(ds),2):
            if count>=PAIR_CAP:break
            count+=1
            for d,side in ((left,'A'),(right,'B')):nn[side+':'+d]={**N[d],'id':side+':'+d,'canonical_id':d,'kind':'Dataset '+side}
            ee.extend([{'s':'A:'+left,'o':h,'relation':'shared '+kind,'layer':'shared_projection','datasets':[left,right],'projection_pair':[left,right],'support_edge_ids':sorted(ds[left])},
                       {'s':h,'o':'B:'+right,'relation':'shared '+kind,'layer':'shared_projection','datasets':[left,right],'projection_pair':[left,right],'support_edge_ids':sorted(ds[right])}])
    PROJECTIONS['Datasets via shared '+kind]={'columns':['Dataset A',kind,'Dataset B'],'nodes':list(nn.values()),'edges':ee,'possibly_capped':count>=PAIR_CAP,'pair_cap':PAIR_CAP}
nd={};ne=[]
for e in E:
    if N[e['s']]['kind']=='Dataset' and N[e['o']]['kind']=='Dataset':
        for d,side in ((e['s'],'A'),(e['o'],'B')):nd[side+':'+d]={**N[d],'id':side+':'+d,'canonical_id':d,'kind':'Dataset '+side}
        ne.append({**e,'s':'A:'+e['s'],'o':'B:'+e['o'],'original_endpoints':{'s':e['s'],'o':e['o']}})
PROJECTIONS['Direct dataset relationships']={'columns':['Dataset A','Dataset B'],'nodes':list(nd.values()),'edges':ne}
GRAPH['projections']=PROJECTIONS
GRAPH['status']={'cohort_datasets':len(COHORT),'cohort_trials':len(NCTS),'exact_catalog_mappings':len(RESOLVED),
                 'cached_trials':len(TRIALS),'cached_publications':len(PUBS),'publication_universe':len(PMIDS),
                 'new_source_calls':dict(CALLS),'capped_entities':len(CAPPED),'failed_entities':len(FAILED)}
# Agreement is source overlap, not independently validated recall or results classification.
pair_layers=defaultdict(set);pair_types=defaultdict(set)
for e in E:
    if e['s'].startswith('pmid:') and e['o'].startswith('nct:'):
        pair_layers[(e['s'],e['o'])].add(e['layer'])
        if e['layer']=='registry_reference':
            for x in e['support']:
                if isinstance(x,dict) and 'registry_type' in x:pair_types[(e['s'],e['o'])].add(x['registry_type'])
rows=[{'publication':p,'clinical_study':n,'layers':';'.join(sorted(v)),'registry_types':';'.join(sorted(pair_types[(p,n)]))} for (p,n),v in sorted(pair_layers.items())]
pd.DataFrame(rows).to_csv(OUT/'publication_trial_source_agreement.csv',index=False)
pending=[]
for ts in list(INDEX.values()):
    for t in ts:
        if t['p'] in STRUCTURAL and t.get('o_type')=='uri' and not recognized(t['o']):pending.append({'predicate':t['p'],'target':t['o']})
pd.DataFrame(pending,columns=['predicate','target']).drop_duplicates().to_csv(OUT/'pending_structural_targets.csv',index=False)
check('Remaining structural coverage',None if pending else True,f'{len(pending)} edge records with unrecognized target typing; see export')
(OUT/'graph.json').write_text(json.dumps(GRAPH,ensure_ascii=False))
(OUT/'http_log.json').write_text(json.dumps(HTTP,indent=2))
pd.DataFrame(CHECKS).to_csv(OUT/'checks.csv',index=False)
award_queue=[{'pmid':p,**g,'meaning':'publication funding acknowledgment; not dataset attribution'} for p,item in PUBS.items() for g in item['record'].get('grants',[])]
pd.DataFrame(award_queue).to_csv(OUT/'publication_award_queue.csv',index=False)
display(pd.DataFrame([GRAPH['status']]));display(pd.DataFrame(CHECKS))

## F. Browser export, debug manifest and final persistence
No Colab widget rendering is required. Open downloaded HTML locally. All column/facet changes redraw automatically. A render counter distinguishes no-match filters from a frozen display; JavaScript errors are visible. Reset restores all evidence, clears facets/highlight, and resets viewport. Use source coverage tables before interpreting missing graph paths.

The eight native classes and Clinical study extension are selectable; zero-instance Collection can be skipped so repository presets remain useful. Preserving an empty Collection heading deliberately creates a gap under strict adjacency.

Drive checkpoints include caches and evidence. Fresh sessions restore automatically. Use MODE='offline' for a cache-only rebuild. Source freshness uses timestamps when available; refresh modes are deliberate retrieval, not display actions. No manual review gates exist.

In [ ]:
# @title
BROWSER_TEMPLATE = '<!DOCTYPE html><html><head><meta charset="utf-8"><title>DATMM consolidated explorer</title><style>*{box-sizing:border-box}body{font-family:Arial;margin:0;background:#f4f6f8;color:#243447}header{padding:12px 18px;background:#16344b;color:white}.shell{display:grid;grid-template-columns:320px minmax(0,1fr);gap:12px;padding:12px;max-width:1920px;margin:auto}.side{height:calc(100vh - 100px);overflow:auto;background:white;padding:12px}.main{position:relative;min-width:0}label{display:block;margin:7px 0;font-size:13px}select{width:100%;padding:5px}.multi{height:105px}button{padding:6px;margin:3px;cursor:pointer}details{border-top:1px solid #ddd;padding-top:7px;margin-top:8px}summary{font-size:13px;font-weight:bold}.slot{display:grid;grid-template-columns:22px 1fr;gap:5px;margin:5px 0}.hint{font-size:12px;line-height:1.4;color:#596775}#chart{height:640px;background:white;margin:10px 0}.bar,#status{background:white;padding:10px;font-size:12px}#errors{color:#a00;background:#fff0f0;font-size:12px;white-space:pre-wrap}.popup{display:none;position:absolute;right:15px;top:75px;width:380px;max-height:500px;overflow:auto;padding:14px;background:white;box-shadow:0 4px 20px #0003;z-index:9}.popup pre{white-space:pre-wrap;overflow-wrap:anywhere;font-size:11px}.popup a{display:block;margin:7px 0;font-size:12px}@media(max-width:1000px){.shell{grid-template-columns:270px minmax(0,1fr)}}</style><script>__LIB__</script></head><body><header><b>DATMM consolidated evidence explorer</b><div style="font-size:12px">Eight native classes + Clinical study • adjacent-column edges • offline filters • no manual curation</div></header><div class="shell"><aside id="controls" class="side"><label>Preset / overlap map</label><select id="preset"></select><label><input id="guided" type="checkbox" checked>Suggest model-connected next columns</label><label><input id="skip" type="checkbox" checked>Skip zero-instance columns</label><div id="slots"></div><div id="warnings" class="hint"></div><label><input id="external" type="checkbox" checked>Include source extensions / registry evidence</label><label><input id="labels" type="checkbox">Labels</label><label><input id="groups" type="checkbox">NIH/NHLBI Agent display aggregation</label><label><input id="generic" type="checkbox">Generic MeSH terms</label><label><input id="focusOnly" type="checkbox">Highlighted neighborhood only</label><label>Evidence layers</label><select id="layers" class="multi" multiple></select><button id="allLayers">All</button><button id="noLayers">Clear layers</button><p class="hint">OR within a facet; AND across enabled facets. Search then select a node to enable its facet. A valid filter can produce zero adjacent edges.</p><div id="facets"></div><button id="reset">Reset all filters and layout</button></aside><main class="main"><div class="bar"><button id="fit">Reset pan/zoom</button><button id="clear">Clear highlight</button><button id="save">Export selected graph</button>Height <input id="height" type="range" min="400" max="850" value="640"></div><div id="errors"></div><div id="chart"></div><div id="status"></div><div id="popup" class="popup"><button id="close" style="float:right">×</button><h3 id="nodeTitle"></h3><div id="links"></div><pre id="data"></pre></div></main></div><script>\nconst S=__DATA__,C={Dataset:\'#4C78A8\',\'Dataset A\':\'#4C78A8\',\'Dataset B\':\'#4C78A8\',Collection:\'#AF7AA1\',Agent:\'#F28E2B\',Concept:\'#59A14F\',Contribution:\'#EDC948\',Documentation:\'#9C755F\',Grant:\'#B279A2\',Repository:\'#76B7B2\',\'Clinical study\':\'#E15759\'};\nconst presets={\'Dataset–Documentation–Clinical study\':[\'Dataset\',\'Documentation\',\'Clinical study\'],\'Agent–Contribution–Dataset–Documentation\':[\'Agent\',\'Contribution\',\'Dataset\',\'Documentation\'],\'Funding + clinical study\':[\'Agent\',\'Contribution\',\'Grant\',\'Dataset\',\'Clinical study\'],\'Grant–Dataset–Clinical study\':[\'Grant\',\'Dataset\',\'Clinical study\'],\'Repository–Collection–Dataset–Documentation\':[\'Repository\',\'Collection\',\'Dataset\',\'Documentation\'],\'Repository–Dataset–Concept\':[\'Repository\',\'Dataset\',\'Concept\'],\'Dataset–Concept–Clinical study\':[\'Dataset\',\'Concept\',\'Clinical study\'],\'All 8 DATMM classes + Clinical study\':[\'Repository\',\'Collection\',\'Dataset\',\'Concept\',\'Documentation\',\'Clinical study\',\'Grant\',\'Contribution\',\'Agent\'],\'Custom\':[\'Dataset\',\'Documentation\',\'Clinical study\']};Object.entries(S.projections).forEach(([k,p])=>presets[k]=p.columns);\nconst el=id=>document.getElementById(id),selection=e=>[...e.selectedOptions].map(o=>o.value),esc=v=>String(v).replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));function opt(select,label,value,on=false){const o=document.createElement(\'option\');o.textContent=label;o.value=value;o.selected=on;select.appendChild(o)}\nconst base=Object.fromEntries(S.nodes.map(n=>[n.id,n])),sourceEdges=Object.fromEntries(S.edges.map(e=>[e.edge_id,e])),adj={};S.classes.forEach(k=>adj[k]=new Set());S.architecture.forEach(([a,b])=>{adj[a].add(b);adj[b].add(a)});function baseKind(k){return k===\'Dataset A\'||k===\'Dataset B\'?\'Dataset\':k}function connected(a,b){return adj[baseKind(a)]&&adj[baseKind(a)].has(baseKind(b))}\nObject.keys(presets).forEach(k=>opt(el(\'preset\'),k,k));[...new Set([...S.edges.map(e=>e.layer),\'shared_projection\'])].sort().forEach(l=>opt(el(\'layers\'),l,l,true));\nconst generic=new Set([\'mesh:D006801\',\'mesh:D008297\',\'mesh:D005260\',\'mesh:D000328\',\'mesh:D000368\',\'mesh:D008875\']);let requested=[...presets[Object.keys(presets)[0]]],projectionName=null,focus=null,last=null,timer,renderNumber=0,currentNodes={},currentEdges=[],ranges=[[-.4,2.4],[-.05,1.15]];\nfunction src(){return projectionName?S.projections[projectionName]:S}function nodeAvailable(n){return projectionName||el(\'external\').checked||n.classification_basis===\'observed native RDF type\'}function classCount(kind){return src().nodes.filter(n=>n.kind===kind&&nodeAvailable(n)).length}\nfunction slots(){el(\'slots\').replaceChildren();const available=projectionName?S.projections[projectionName].columns:S.classes;for(let i=0;i<9;i++){let row=document.createElement(\'div\');row.className=\'slot\';let number=document.createElement(\'span\');number.textContent=i+1;let select=document.createElement(\'select\');select.className=\'column-slot\';select.dataset.slot=i;opt(select,\'— no column —\',\'\',!requested[i]);let previous=i?requested[i-1]:null;let choices=el(\'guided\').checked&&previous&&!projectionName?available.filter(c=>connected(previous,c)):available;choices=[...new Set([...choices,...(requested[i]?[requested[i]]:[])])];choices.forEach(c=>opt(select,c+\' (\'+classCount(c)+\')\'+(previous&&!connected(previous,c)&&!projectionName?\' — no direct model link\':\'\'),c,c===requested[i]));row.append(number,select);el(\'slots\').appendChild(row)}let empty=requested.filter(c=>classCount(c)===0);el(\'warnings\').textContent=(empty.length?\'Zero-instance columns: \'+empty.join(\', \')+(el(\'skip\').checked?\' — skipped.\':\' — retained.\'):\'\')+(projectionName?\' Dataset A/B are pair members, not different classes.\':\' DATMM is branched; not every column order forms a populated path.\')}\nconst facets={};for(let kind of S.classes){let box=document.createElement(\'details\');box.dataset.kind=kind;let header=document.createElement(\'summary\');header.textContent=kind+\' facet\';box.appendChild(header);let enabled=document.createElement(\'input\');enabled.type=\'checkbox\';enabled.className=\'facet-enable\';let lab=document.createElement(\'label\');lab.append(enabled,document.createTextNode(\' Enable filter\'));box.appendChild(lab);let search=document.createElement(\'input\');search.type=\'text\';search.className=\'facet-search\';search.placeholder=\'Search \'+kind;search.style.width=\'100%\';box.appendChild(search);let select=document.createElement(\'select\');select.multiple=true;select.className=\'multi facet-choice\';S.nodes.filter(n=>n.kind===kind).sort((a,b)=>a.label.localeCompare(b.label)).forEach(n=>opt(select,n.label+(n.subtype?\' [\'+n.subtype+\']\':\'\'),n.id));box.appendChild(select);let off=document.createElement(\'button\');off.className=\'facet-disable\';off.textContent=\'Disable\';box.appendChild(off);el(\'facets\').appendChild(box);facets[kind]={enabled,search,select}}\nfunction reportError(error){el(\'errors\').textContent=\'Render error: \'+(error.stack||error.message||String(error))}function schedule(){clearTimeout(timer);timer=setTimeout(()=>draw().catch(reportError),120)}\nel(\'controls\').addEventListener(\'change\',event=>{let t=event.target;if(t.id===\'preset\'){projectionName=S.projections[t.value]?t.value:null;requested=[...presets[t.value]];if(t.value.startsWith(\'All 8\'))el(\'guided\').checked=false;focus=null;slots()}\nelse if(t.classList.contains(\'column-slot\')){let i=Number(t.dataset.slot);if(!t.value)requested=requested.slice(0,i);else{const already=requested.indexOf(t.value);if(already>=0&&already!==i){const previous=requested[i];requested[i]=t.value;if(previous)requested[already]=previous;else requested.splice(already,1)}else requested[i]=t.value;requested=requested.filter(Boolean)}el(\'preset\').value=\'Custom\';focus=null;slots()}\nelse if(t.classList.contains(\'facet-choice\'))facets[t.closest(\'[data-kind]\').dataset.kind].enabled.checked=true;\nif([\'guided\',\'skip\',\'external\'].includes(t.id))slots();schedule()});el(\'facets\').addEventListener(\'input\',event=>{if(!event.target.classList.contains(\'facet-search\'))return;const f=facets[event.target.closest(\'[data-kind]\').dataset.kind],q=f.search.value.toLowerCase();[...f.select.options].forEach(o=>o.hidden=!o.textContent.toLowerCase().includes(q))});el(\'facets\').addEventListener(\'click\',event=>{if(event.target.classList.contains(\'facet-disable\')){facets[event.target.closest(\'[data-kind]\').dataset.kind].enabled.checked=false;schedule()}});\nfunction canonical(n){return n.canonical_id||n.id}function popup(n){el(\'popup\').style.display=\'block\';el(\'nodeTitle\').textContent=n.label;el(\'links\').replaceChildren();let urls=[],id=canonical(n);for(const alias of n.aliases||[]){const match=alias.match(/^https?:\\/\\/id\\.nlm\\.nih\\.gov\\/datmm\\/dataset\\/(\\d+)$/);if(match)urls.push([\'NLM Dataset Catalog record\',\'https://datasetcatalog.nlm.nih.gov/dataset?q=\'+match[1]])}if(!urls.length)urls.push([\'Search NLM Dataset Catalog\',\'https://datasetcatalog.nlm.nih.gov/?q=\'+encodeURIComponent(n.label)]);if(id.startsWith(\'pmid:\'))urls.push([\'PubMed\',\'https://pubmed.ncbi.nlm.nih.gov/\'+id.slice(5)]);if(id.startsWith(\'nct:\'))urls.push([\'ClinicalTrials.gov\',\'https://clinicaltrials.gov/study/\'+id.slice(4)]);if(id.startsWith(\'dbgap:\'))urls.push([\'dbGaP\',\'https://www.ncbi.nlm.nih.gov/projects/gap/cgi-bin/study.cgi?study_id=\'+encodeURIComponent(id.slice(6))]);if(id.startsWith(\'mesh:\'))urls.push([\'MeSH RDF identifier\',\'https://id.nlm.nih.gov/mesh/\'+id.slice(5)]);for(let [label,url] of [...new Map(urls.map(x=>[x[1],x])).values()]){if(!/^https?:\\/\\//.test(url))continue;let a=document.createElement(\'a\');a.textContent=label;a.href=url;a.target=\'_blank\';a.rel=\'noopener noreferrer\';el(\'links\').appendChild(a)}let edges=currentEdges.filter(e=>e.s===n.id||e.o===n.id);el(\'data\').textContent=JSON.stringify({node:n,incident_edges:edges,projection_source_edges:edges.flatMap(e=>(e.support_edge_ids||[]).map(id=>sourceEdges[id]).filter(Boolean))},null,2)}\nasync function draw(){const rid=++renderNumber;el(\'errors\').textContent=\'\';el(\'status\').textContent=\'Rendering \'+rid+\'…\';let source=src(),order=el(\'skip\').checked?requested.filter(c=>classCount(c)>0):[...requested];if(!order.length){Plotly.purge(\'chart\');el(\'status\').textContent=\'No available columns; change preset or include source extensions.\';return}let layers=new Set(selection(el(\'layers\'))),nodes=Object.fromEntries(source.nodes.map(n=>[n.id,{...n}])),ds=new Set(S.datasets),restrictions={},filterText=[];\nfor(let [kind,f] of Object.entries(facets)){if(!f.enabled.checked)continue;let ids=new Set(selection(f.select));restrictions[kind]=ids;let allowed=new Set();ids.forEach(u=>(base[u].datasets||[]).forEach(d=>allowed.add(d)));ds=new Set([...ds].filter(d=>allowed.has(d)));filterText.push(kind+\': \'+ids.size)}\nlet candidates=source.edges.filter(e=>layers.has(e.layer)&&nodes[e.s]&&nodes[e.o]&&nodeAvailable(nodes[e.s])&&nodeAvailable(nodes[e.o])&&order.includes(nodes[e.s].kind)&&order.includes(nodes[e.o].kind)).map(e=>({...e,datasets:(e.datasets||[]).filter(d=>ds.has(d))})).filter(e=>e.datasets.length&&(!e.projection_pair||e.projection_pair.every(d=>ds.has(d)))&&(el(\'generic\').checked||(!generic.has(e.s)&&!generic.has(e.o)))).filter(e=>[e.s,e.o].every(u=>!restrictions[baseKind(nodes[u].kind)]||restrictions[baseKind(nodes[u].kind)].has(canonical(nodes[u]))));\nlet hidden=candidates.filter(e=>Math.abs(order.indexOf(nodes[e.s].kind)-order.indexOf(nodes[e.o].kind))!==1).length,edges=candidates.filter(e=>Math.abs(order.indexOf(nodes[e.s].kind)-order.indexOf(nodes[e.o].kind))===1);\nif(!projectionName&&el(\'groups\').checked)edges=edges.map(original=>{let e={...original};for(let k of [\'s\',\'o\']){let g=S.organization_display_groups[e[k]];if(g){e.original_endpoints=e.original_endpoints||{s:e.s,o:e.o};nodes[g.group]={id:g.group,kind:\'Agent\',label:g.label,display_rule:g};e[k]=g.group}}return e});const dedup=new Map();edges.forEach(e=>{let k=JSON.stringify([e.s,e.o,e.relation,e.layer]);if(!dedup.has(k))dedup.set(k,{...e,source_records:[e]});else{let a=dedup.get(k);a.source_records.push(e);a.datasets=[...new Set([...a.datasets,...e.datasets])]}});edges=[...dedup.values()];let active=new Set(edges.flatMap(e=>[e.s,e.o])),selectedIds=new Set([...active].filter(u=>focus&&canonical(nodes[u])===focus)),near=new Set(selectedIds);edges.forEach(e=>{if(selectedIds.has(e.s)||selectedIds.has(e.o)){near.add(e.s);near.add(e.o)}});if(el(\'focusOnly\').checked&&selectedIds.size){edges=edges.filter(e=>selectedIds.has(e.s)||selectedIds.has(e.o));active=new Set(edges.flatMap(e=>[e.s,e.o]))}let pos={};order.forEach((kind,i)=>{let ids=[...active].filter(u=>nodes[u].kind===kind).sort((a,b)=>nodes[a].label.localeCompare(nodes[b].label));ids.forEach((u,j)=>pos[u]=[i,(j+1)/(ids.length+1)])});let traces=[],buckets=new Map();edges.forEach(e=>{let hot=selectedIds.size&&(selectedIds.has(e.s)||selectedIds.has(e.o)),dim=selectedIds.size&&!hot,k=e.layer+\'|\'+(dim?\'dim\':hot?\'hot\':\'normal\');if(!buckets.has(k))buckets.set(k,{edges:[],hot,dim,layer:e.layer});buckets.get(k).edges.push(e)});for(let b of buckets.values()){let x=[],y=[],text=[];b.edges.forEach(e=>{x.push(pos[e.s][0],pos[e.o][0],null);y.push(pos[e.s][1],pos[e.o][1],null);text.push(esc(e.relation),esc(e.relation),\'\')});traces.push({type:\'scatter\',mode:\'lines\',x,y,text,showlegend:false,opacity:b.dim?0.1:1,line:{width:b.hot?2.7:1,color:b.hot?\'#174a73\':\'#89939e\',dash:b.layer===\'native_datmm\'?\'solid\':\'dash\'},hovertemplate:\'%{text}<extra></extra>\'})}\nfor(let kind of order){let ids=[...active].filter(u=>nodes[u].kind===kind);if(ids.length)traces.push({type:\'scatter\',mode:el(\'labels\').checked?\'markers+text\':\'markers\',name:kind,x:ids.map(u=>pos[u][0]),y:ids.map(u=>pos[u][1]),text:ids.map(u=>esc(nodes[u].label.slice(0,65))),textposition:\'top center\',marker:{size:ids.map(u=>selectedIds.has(u)?19:11),color:C[kind],opacity:ids.map(u=>selectedIds.size&&!near.has(u)?0.15:1)},customdata:ids,hovertext:ids.map(u=>esc(nodes[u].label)+\'<br>\'+esc(nodes[u].subtype||kind)+\'<br>\'+esc(nodes[u].classification_basis)),hovertemplate:\'%{hovertext}<extra></extra>\'})}\nlet annotations=order.map((kind,i)=>({x:i,y:1.10,text:kind+\' (\'+[...active].filter(u=>nodes[u].kind===kind).length+\')\',showarrow:false}));if(!edges.length)annotations.push({x:0,y:.5,text:\'Filter applied: no adjacent edges match. Disable a facet or reset.\',showarrow:false});ranges=[[-.4,Math.max(.6,order.length-.6)],[-.05,1.15]];currentNodes=nodes;currentEdges=edges;last={requested_columns:requested,rendered_columns:order,nodes:[...active].map(u=>nodes[u]),edges,filters:filterText,coverage:S.status};let height=Number(el(\'height\').value);el(\'chart\').style.height=height+\'px\';await Plotly.react(\'chart\',traces,{title:{text:(projectionName||\'Model + clinical study\')+\': \'+order.join(\' → \'),font:{size:14}},height,dragmode:\'pan\',uirevision:(projectionName||\'native\')+\'|\'+order.join(\'|\'),xaxis:{visible:false,range:ranges[0],fixedrange:false},yaxis:{visible:false,range:ranges[1],fixedrange:false},annotations,margin:{t:80,b:45,l:20,r:20},legend:{orientation:\'h\'}},{responsive:true,displaylogo:false,scrollZoom:true});if(rid!==renderNumber)return;el(\'chart\').removeAllListeners(\'plotly_click\');el(\'chart\').on(\'plotly_click\',event=>{let u=event.points[0].customdata;if(!nodes[u])return;let target=canonical(nodes[u]);focus=focus===target?null:target;if(focus)popup(nodes[u]);else el(\'popup\').style.display=\'none\';schedule()});el(\'status\').textContent=`Render ${rid}: ${active.size} nodes; ${edges.length} adjacent edges; ${ds.size} eligible cohort datasets; ${hidden} non-adjacent records hidden. Filters: ${filterText.join(\'; \')||\'none\'}. Skipped columns: ${requested.filter(c=>!order.includes(c)).join(\', \')||\'none\'}. ${projectionName&&source.possibly_capped?\'Projection pair cap reached. \':\'\'}No browser source requests.`}\nwindow.addEventListener(\'error\',e=>reportError(e.error||e.message));window.addEventListener(\'unhandledrejection\',e=>reportError(e.reason));el(\'height\').oninput=schedule;el(\'fit\').onclick=()=>Plotly.relayout(\'chart\',{\'xaxis.range\':ranges[0],\'yaxis.range\':ranges[1]});el(\'clear\').onclick=()=>{focus=null;el(\'popup\').style.display=\'none\';schedule()};el(\'close\').onclick=()=>el(\'popup\').style.display=\'none\';el(\'allLayers\').onclick=()=>{[...el(\'layers\').options].forEach(o=>o.selected=true);schedule()};el(\'noLayers\').onclick=()=>{[...el(\'layers\').options].forEach(o=>o.selected=false);schedule()};el(\'reset\').onclick=()=>{el(\'preset\').selectedIndex=0;requested=[...presets[el(\'preset\').value]];projectionName=null;el(\'guided\').checked=true;el(\'skip\').checked=true;el(\'external\').checked=true;el(\'labels\').checked=false;el(\'groups\').checked=false;el(\'generic\').checked=false;el(\'focusOnly\').checked=false;el(\'height\').value=640;[...el(\'layers\').options].forEach(o=>o.selected=true);for(let f of Object.values(facets)){f.enabled.checked=false;f.search.value=\'\';[...f.select.options].forEach(o=>{o.selected=false;o.hidden=false})}focus=null;el(\'popup\').style.display=\'none\';slots();draw().then(()=>Plotly.relayout(\'chart\',{\'xaxis.range\':ranges[0],\'yaxis.range\':ranges[1]})).catch(reportError)};el(\'save\').onclick=()=>{if(!last)return;let a=document.createElement(\'a\');a.href=URL.createObjectURL(new Blob([JSON.stringify(last,null,2)],{type:\'application/json\'}));a.download=\'datmm_selection.json\';a.click();setTimeout(()=>URL.revokeObjectURL(a.href),1000)};slots();draw().catch(reportError);\n</script></body></html>'
page=BROWSER_TEMPLATE.replace('__LIB__',get_plotlyjs()).replace('__DATA__',json.dumps(GRAPH,ensure_ascii=False).replace('<',chr(92)+'u003c'))
browser_path=OUT/'datmm_consolidated_browser.html';browser_path.write_text(page,encoding='utf-8')
MANIFEST={'created_utc':datetime.now(timezone.utc).isoformat(),'mode':MODE,'budgets':BUDGETS,'calls':dict(CALLS),
          'source_coverage':GRAPH['status'],'class_counts':NATIVE_CLASS_COUNTS,'capped_entities':sorted(CAPPED),
          'failed_entities':sorted(FAILED),'checks':dict(Counter(x['status'] for x in CHECKS)),
          'manual_review_required':False,'browser_source_requests':0,
          'limitations':['No RePORTER award-year resolution','Bounded source extraction','No reuse or causal funding assertion',
                         'Organization grouping is display aggregation','Native normalization can reduce duplicate instance counts',
                         'Shared-projection pair cap','Clinical study is an application extension']}
(OUT/'manifest.json').write_text(json.dumps(MANIFEST,indent=2))
final_checkpoint=checkpoint('consolidated-final')
outputs_zip=shutil.make_archive(str(WORK.parent/'datmm_consolidated_outputs'),'zip',root_dir=OUT)
if DRIVE_ROOT:
    target=DRIVE_ROOT/'browser_exports';target.mkdir(exist_ok=True);shutil.copy2(browser_path,target/browser_path.name)
if IN_COLAB:
    from google.colab import files
    files.download(str(browser_path));files.download(outputs_zip)
else:
    print('Browser:', browser_path.resolve())
    print('Output archive:', Path(outputs_zip).resolve())
print('Full cache checkpoint saved separately:',Path(final_checkpoint).name)

## Troubleshooting checklist
| Symptom | Check/action |
|---|---|
| No prior snapshot | Fresh acquisition or embedded-source graph is supported; no old notebook needed. |
| No typed funder/contributor branch | Check structural budget and pending_structural_targets.csv; cache_first resumes missing work. |
| Collection zero | Do not invent instances; skip empty columns or preserve zero heading explicitly. |
| Facet seems unchanged | Watch Render number/filter summary; no-match and JS errors are reported. |
| Empty native-only view | Source-layer nodes require the extension toggle; inspect class_coverage.csv. |
| Many hidden edges | Strict adjacency depends on column order; use a model-connected preset. |
| Blank Colab plot | Browser HTML is the supported display path. |
| Dictionary mutation | Non-mutating reads, snapshot iteration, and regression test prevent the prior bug. |
| 500/504 | Logged as retrieval failures; do not infer no metadata; reuse cache/retry with budget. |
| Different class counts from earlier | This notebook normalizes exact MeSH/PMID/accession identities while preserving source aliases. |

To resume: run again in cache_first mode. To render only: offline mode with cached state. Optional future adapters can resolve the publication award queue, but original funding identifiers remain separate from validated NIH award-year entities.

## G. Optional versioned GitHub release — disabled by default

You may run this cell independently to package the supplied snapshot, without executing acquisition cells. It loads `scripts/datmm_release.py` from the accompanying project directory. In Colab, if that directory is absent, `prepare` asks for the starter ZIP. See `RELEASE_GUIDE.md` for the one-time repository/token setup and GitHub Pages instructions.

1. Use `ACTION = "prepare"`. Set the version and explicit notebook/browser/extra-file paths. Inspect the frozen files in `dist/<version>/package`.
2. Use `ACTION = "publish"` to publish that already prepared package. Type the repository/version when prompted. Only this action reads the `GITHUB_DATMM` secret.
3. Return `ACTION` to `"off"` afterward. Packaged notebook copies automatically reset this action to `"off"`.

**Colab editor changes are not runtime files.** To publish your edited notebook, download its `.ipynb`, upload it into the runtime Files panel, and set `NOTEBOOK_SOURCE` to that file. Leaving the path as `None` publishes the notebook from the extracted package. `BROWSER_SOURCE=None` uses the supplied browser, even if a new acquisition ran earlier. Set both paths explicitly for a new evidentiary run.

The publisher includes no Drive trees, caches, checkpoint ZIPs, or extra outputs automatically. The manifest lists only selected files. A successful release establishes artifact identity and availability, not independent verification of the graph's scientific assertions.


In [ ]:
# @title Optional GitHub release (off / prepare / publish)
ACTION = "off"  # off, prepare, publish
VERSION = "v0.1.1"
REPOSITORY = "conroyriodss/datmm-explorer"
BRANCH = "main"
PRERELEASE = False
PROJECT_DIR = None  # e.g. "/content/datmm-release"; auto-locate when None.
NOTEBOOK_SOURCE = None  # Full path to the saved .ipynb you intend to release.
BROWSER_SOURCE = None  # None = supplied docs/index.html; set explicitly for new output.
EXTRA_FILES = {}  # e.g. {"exports/checks.csv": "/content/datmm_next/consolidated/output/checks.csv"}
RELEASE_NOTES = "Initial packaging of the supplied DATMM notebook and browser snapshot; no new source acquisition."

if ACTION == "off":
    print("GitHub release is off. No files, credentials, or network were accessed by this cell.")
elif ACTION not in {"prepare", "publish"}:
    raise ValueError("ACTION must be off, prepare, or publish")
else:
    from pathlib import Path
    import importlib.util
    import shutil
    import tempfile
    import zipfile

    if PROJECT_DIR is None:
        candidates = [Path.cwd(), Path.cwd().parent, Path("/content/datmm-release")]
        project = next((p for p in candidates if (p / "scripts/datmm_release.py").is_file()), None)
    else:
        project = Path(PROJECT_DIR).expanduser().resolve()
    if project is None or not (project / "scripts/datmm_release.py").is_file():
        if ACTION == "publish":
            raise FileNotFoundError("Prepared project is missing. Run prepare first and keep its dist folder.")
        try:
            from google.colab import files as colab_files
        except ImportError:
            raise FileNotFoundError("Set PROJECT_DIR to the extracted datmm-release directory.") from None
        print("Upload datmm-github-release-starter.zip to provide the companion files.")
        uploaded = colab_files.upload()
        if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith(".zip"):
            raise ValueError("Upload exactly the starter ZIP.")
        archive_path = Path(next(iter(uploaded)))
        base = Path("/content")
        target = base / "datmm-release"
        if target.exists():
            raise FileExistsError("/content/datmm-release already exists; select it explicitly or inspect it before replacing.")
        temporary = Path(tempfile.mkdtemp(prefix="datmm-unpack-", dir=base))
        try:
            with zipfile.ZipFile(archive_path) as archive:
                if sum(i.file_size for i in archive.infolist()) > 250_000_000:
                    raise ValueError("Starter ZIP is unexpectedly large.")
                for info in archive.infolist():
                    relative = Path(info.filename)
                    destination = (temporary / relative).resolve()
                    if (not relative.parts or relative.parts[0] != "datmm-release"
                            or relative.is_absolute() or ".." in relative.parts
                            or "\\" in info.filename or temporary not in destination.parents
                            or ((info.external_attr >> 16) & 0o170000) == 0o120000):
                        raise ValueError("Unsafe ZIP entry rejected.")
                archive.extractall(temporary)
            (temporary / "datmm-release").rename(target)
        finally:
            shutil.rmtree(temporary, ignore_errors=True)
        project = target
        del uploaded
    project = project.resolve()
    spec = importlib.util.spec_from_file_location("datmm_release", project / "scripts/datmm_release.py")
    publisher = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(publisher)
    if ACTION == "prepare":
        print("Notebook selected:", Path(NOTEBOOK_SOURCE) if NOTEBOOK_SOURCE else project / publisher.NOTEBOOK)
        print("Browser selected:", Path(BROWSER_SOURCE) if BROWSER_SOURCE else project / "docs/index.html")
        BUILD = publisher.prepare(project, VERSION, notebook_path=NOTEBOOK_SOURCE,
                                  browser_path=BROWSER_SOURCE, extra_files=EXTRA_FILES,
                                  notes=RELEASE_NOTES)
    else:
        BUILD = project / "dist" / VERSION
        frozen, _, _ = publisher.verify(BUILD)
        print("Publishing the existing frozen package, not new editor/runtime changes:", BUILD)
        print("Files become public when the repository branch is updated.")
        confirmation = input(f"Type {REPOSITORY}@{VERSION} to publish: ").strip()
        if confirmation != f"{REPOSITORY}@{VERSION}":
            raise ValueError("Publication cancelled; no token was read.")
        token = publisher.get_token()
        try:
            RELEASE_RESULT = publisher.publish(BUILD, REPOSITORY, token, branch=BRANCH,
                                               confirmation=confirmation, prerelease=PRERELEASE)
        finally:
            del token
